# Before you launch the job

Five checks, in the order you would actually do them.

**This page needs a GPU.**  `Runtime > Change runtime type > T4 GPU`, then
`Runtime > Run all`.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/infer.py', 'lib/preflight.py', 'lib/sr_models.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'infer', 'preflight', 'sr_models'):
    sys.modules.pop(m, None)

import numpy as np
import torch
from infer import *
from preflight import *
from sr_models import EDSR, load_edsr

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
PATH = get('dataset/condition/clear.tif', 'scene.tif')
WEIGHT = get('models/14_inference/checkpoints/edsr_x2.pt', 'edsr_x2.pt')

## Q1.  Is there a GPU, and what is it?

In [ ]:
print('cuda available', torch.cuda.is_available())
print()
gpu_state()

`memory` is the number that decides the tile and batch sizes further down.
`tensor cores` decides whether `fp16` is actually faster or only smaller.

## Q2.  Convert the input to what the model expects

The scene is 12-bit in a `uint16` container.  A model trained on 8-bit images expects
0 - 255; one trained on 16-bit expects 0 - 65535.  Both come from the same window — a
percentile range of the original values — stretched onto a different ceiling.

In [ ]:
a16, window = to_bits(PATH, bits=16)
a8, _ = to_bits(PATH, bits=8)

for i, (lo, hi) in enumerate(zip(*window), start=1):
    print(f'band {i}   2 - 99.5 % window   {lo:.0f} - {hi:.0f}')
print()
print('16-bit', a16.shape, a16.dtype, a16.min(), '-', a16.max())
print(' 8-bit', a8.shape, a8.dtype, a8.min(), '-', a8.max())

In [ ]:
show_inputs(PATH)

The original sits in a narrow band with a long thin tail.  Both conversions spread
that band across the whole range, which is the point — and both put a spike at each
end, which is everything that fell outside the window.

Keep `window`.  Without it the result cannot be turned back into sensor units.

## Q3.  Cut the scene into patches

`tile` is what each patch keeps.  `overlap` is the margin read around it and thrown
away, so that pixels near a tile edge still had neighbours when the network saw
them.

In [ ]:
cfg = Config(scale=2, tile=256, overlap=200, batch=4, precision='fp32')
plan(*a8.shape[:2], cfg)

In [ ]:
show_patches(PATH, cfg)

Blue is what each tile keeps; the dashed orange box is what one tile actually reads.

An overlap of 200 around a tile of 256 means every patch reads 656 pixels a side to
keep 256 — the network runs over **ten times** the scene.  That is the cost of a margin
chosen far larger than it needs to be; 16 would do for this model.

## Q4.  Precision and batch size

`fp32` is the safe default.  `fp16` halves the memory every activation takes and runs
faster where tensor cores exist; on this model the output differs by at most one
level.  `batch` is how many patches go in one forward pass.

In [ ]:
net = load_edsr(WEIGHT, scale=2)
sr_tiled(a8[:128, :128], net, cfg)      # first call pays for CUDA start-up

cfg = replace(cfg, overlap=16, batch=4, precision='fp32')
try_run(a8, net, cfg)

cfg16 = replace(cfg, precision='fp16')
try_run(a8, net, cfg16)

## Q5.  The scripts

### Training

Enough iterations to prove the script runs — not enough to train anything.  Real
weights come from a full run on a server.

In [ ]:
import cv2
import torch.nn as nn

SCALE, PATCH, BS = 2, 96, 8
hr = a8                                             # 8-bit scene as the target
lo = cv2.resize(hr, (hr.shape[1] // SCALE, hr.shape[0] // SCALE),
                interpolation=cv2.INTER_CUBIC)      # its own input

model = EDSR(scale=SCALE).to(cfg.device).train()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.L1Loss()
rng = np.random.default_rng(0)

for step in range(20):
    y = rng.integers(0, lo.shape[0] - PATCH, BS)
    x = rng.integers(0, lo.shape[1] - PATCH, BS)
    a = np.stack([lo[y[k]:y[k] + PATCH, x[k]:x[k] + PATCH] for k in range(BS)])
    b = np.stack([hr[y[k] * SCALE:(y[k] + PATCH) * SCALE,
                     x[k] * SCALE:(x[k] + PATCH) * SCALE] for k in range(BS)])
    a = torch.from_numpy(a).permute(0, 3, 1, 2).float().to(cfg.device)
    b = torch.from_numpy(b).permute(0, 3, 1, 2).float().to(cfg.device)

    opt.zero_grad(set_to_none=True)
    loss = loss_fn(model(a), b)
    loss.backward()
    opt.step()
    if step % 5 == 0:
        print(f'step {step:3d}   L1 {loss.item():6.2f}')

### Inference

Convert, run tiled, convert back, write with the input's georeferencing.

In [ ]:
lr8, (lo_w, hi_w) = to_bits(PATH, bits=8)
sr = sr_tiled(lr8, net, cfg, report=True)

sr12 = np.clip(sr / 255 * (hi_w - lo_w) + lo_w, 0, 65535).round().astype('uint16')
OUT = save_sr(PATH, sr12, 'sr.tif', scale=cfg.scale)
print('wrote', OUT, sr12.shape, sr12.dtype)

The output covers exactly the same ground as the input, with pixels half the size.
Checking that it really does is the next page.